<a href="https://colab.research.google.com/github/HITESH8842/dataviz-exercises-M-Hitesh/blob/main/week04_lecture04_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 4 — Class Exercise
## Scatter & Bubble Charts: Gapminder

> **Push to:** `week04/lecture04_exercise.ipynb`

**Rules:**
1. Colour used **sparingly** — one categorical variable, no rainbow
2. If showing all continents, either use accessible palette OR grey all + highlight one
3. `size_max` set when using bubble size
4. Log scale for GDP per capita
5. Insight title


In [ ]:
import pandas as pd
import plotly.express as px


# Dataset: Gapminder — GDP, Life Expectancy, Population by Country
# Source: Gapminder Foundation (gapminder.org)

df = px.data.gapminder()
print(f"Loaded: {len(df)} rows")
print(df.head())

Loaded: 1704 rows
       country continent  year  lifeExp       pop   gdpPercap iso_alpha  \
0  Afghanistan      Asia  1952   28.801   8425333  779.445314       AFG   
1  Afghanistan      Asia  1957   30.332   9240934  820.853030       AFG   
2  Afghanistan      Asia  1962   31.997  10267083  853.100710       AFG   
3  Afghanistan      Asia  1967   34.020  11537966  836.197138       AFG   
4  Afghanistan      Asia  1972   36.088  13079460  739.981106       AFG   

   iso_num  
0        4  
1        4  
2        4  
3        4  
4        4  


In [ ]:
# explore

print(df.info())
print("Years:", sorted(df['year'].unique()))
print("Continents:", df['continent'].unique())
print(df.describe().round(1))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1704 entries, 0 to 1703
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   country    1704 non-null   object 
 1   continent  1704 non-null   object 
 2   year       1704 non-null   int64  
 3   lifeExp    1704 non-null   float64
 4   pop        1704 non-null   int64  
 5   gdpPercap  1704 non-null   float64
 6   iso_alpha  1704 non-null   object 
 7   iso_num    1704 non-null   int64  
dtypes: float64(2), int64(3), object(3)
memory usage: 106.6+ KB
None
Years: [np.int64(1952), np.int64(1957), np.int64(1962), np.int64(1967), np.int64(1972), np.int64(1977), np.int64(1982), np.int64(1987), np.int64(1992), np.int64(1997), np.int64(2002), np.int64(2007)]
Continents: ['Asia' 'Europe' 'Africa' 'Americas' 'Oceania']
         year  lifeExp           pop  gdpPercap  iso_num
count  1704.0   1704.0  1.704000e+03     1704.0   1704.0
mean   1979.5     59.5  2.960121e+07     7215.3    

## Task 1 — Scatter: life expectancy change over time

**What to build:** A scatter showing **GDP per capita vs life expectancy** for **two years** (2002 and 2007) to show how both moved — use **colour for year** (just 2 colours), **one continent only**.

Choose any continent except Africa (that was the example). Highlight the change direction.

> 💡 Filter: `df.loc[df['continent'] == 'YOUR_CHOICE']` then filter years


In [ ]:
# Task 1
import numpy as np

t1 = df.loc[(df["continent"] == "Asia") & (df["year"].isin([2002, 2007]))].copy()
t1["year"] = t1["year"].astype(str)  # 2 discrete colours, not a colour bar

fig1 = px.scatter(
    t1, x="gdpPercap", y="lifeExp", color="year", hover_name="country",
    color_discrete_map={"2002": "#B0B0B0", "2007": "#1F77B4"},  # grey = before, blue = after
    log_x=True,
    title="<b>Every Asian country gained life expectancy 2002→2007 (avg +1.5 yrs) as GDP rose</b>",
    labels={"gdpPercap": "GDP per capita (log)", "lifeExp": "Life expectancy", "year": "Year"},
    template="plotly_white",
)

# arrow from each country's 2002 point to its 2007 point = change direction
# (on a log axis, annotation x-coords must be log10 of the value)
for _, g in t1.sort_values("year").groupby("country"):
    fig1.add_annotation(x=np.log10(g.gdpPercap.iloc[1]), y=g.lifeExp.iloc[1],
                        ax=np.log10(g.gdpPercap.iloc[0]), ay=g.lifeExp.iloc[0],
                        axref="x", ayref="y", arrowhead=2, arrowwidth=1,
                        arrowcolor="rgba(31,119,180,0.35)", text="")

fig1.update_traces(marker=dict(size=10))
fig1.show()

## Task 2 — Bubble chart: tell a story

**What to build:** A bubble chart (full 2007 dataset, all countries) where:
- x = GDP per capita (log scale)
- y = life expectancy
- size = population
- colour = ONE continent highlighted (your choice), all others grey
- At least one annotation explaining the highlighted group's story

> This is the grey-and-highlight technique applied to a bubble chart.

In [ ]:
# Task 2
import numpy as np

t2 = df.loc[df["year"] == 2007].copy()
t2["group"] = t2["continent"].where(t2["continent"] == "Asia", "Other")  # 2 colours only

fig2 = px.scatter(
    t2, x="gdpPercap", y="lifeExp", size="pop", size_max=60, color="group",
    color_discrete_map={"Asia": "#D62728", "Other": "#D3D3D3"},
    category_orders={"group": ["Other", "Asia"]},  # grey drawn first, red on top
    hover_name="country", log_x=True,
    title="<b>Asia holds 61% of the world's people but spans 39 years of life expectancy</b>"
          "<br><sup>2007 · bubble size = population · grey = other continents</sup>",
    labels={"gdpPercap": "GDP per capita (log)", "lifeExp": "Life expectancy", "group": ""},
    template="plotly_white",
)

# annotations (x must be log10 on a log axis)
def pt(name):
    r = t2.loc[t2.country == name].iloc[0]
    return np.log10(r.gdpPercap), r.lifeExp

for name, text, ax, ay in [
    ("China",       "<b>China</b> — 1.3 bn, 73 yrs",            -110,  80),
    ("India",       "<b>India</b> — 1.1 bn, under $2.5k, 65 yrs", -20, 110),
    ("Japan",       "<b>Japan</b> — world's highest, 82.6 yrs",   -60, -60),
    ("Afghanistan", "<b>Afghanistan</b> — $975, 44 yrs",           60, -10),
]:
    x, y = pt(name)
    fig2.add_annotation(x=x, y=y, text=text, ax=ax, ay=ay, arrowhead=2,
                        bgcolor="white", bordercolor="#D62728", font=dict(size=11))

fig2.update_traces(marker=dict(opacity=0.75))
fig2.show()